# ☀️ ICARUS_WORKER_HELIOPHYSICS — Heliophysics & Space Weather Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 3/4)

**Role:** Heliophysics, Solar Dynamics & Space Weather Forecasting  
**Primary Model:** NASA-IBM Surya-1.0 (`nasa-ibm-ai4science/Surya-1.0`)  
**Checkpoint:** `surya.366m.v1.pt` (366M parameters)  
**Data Sources:** NASA Solar Dynamics Observatory (SDO) AIA EUV and HMI products  
**Deployment Profile:** Kaggle GPU (T4 or P100)

---

### Worker Responsibilities
- Hosts deterministic heliophysics algorithms: **GOES flare classification, Kp/Dst storm categorization, solar differential rotation**.
- Hosts **NASA-IBM Surya-1.0** for solar spatiotemporal representation extraction.
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


In [ ]:
# 1. Environment & Dependencies Setup
import os
import sys
from pathlib import Path

!pip install -q huggingface_hub sunpy astropy pyngrok uvicorn fastapi pydantic

!git clone https://github.com/kanishk10-ux/ICARUS.git /kaggle/working/ICARUS 2>/dev/null || (cd /kaggle/working/ICARUS && git pull)
sys.path.insert(0, "/kaggle/working/ICARUS")

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Surya-1.0 Adapter Verification & Smoke Test
import numpy as np
from icarus.workers.heliophysics.surya_adapter import SuryaSolarAdapter
from icarus.algorithms.heliophysics import HeliophysicsAlgorithms

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = SuryaSolarAdapter(device=device)

ok, msg = adapter.load_model()
print("Load Status:", ok, "->", msg)

# 8 AIA EUV channels of 224x224 pixels
dummy_solar_obs = np.random.uniform(1.0, 500.0, (8, 224, 224)).astype(np.float32)
res = adapter.extract_features(dummy_solar_obs, observation_time_iso="2024-05-10T12:00:00Z")
print("Surya Output Status:", res["status"])
print(f"Feature Shape: {res['feature_shape']} ({res['total_patch_tokens']} tokens x {res['hidden_dimension']} dims)")

# Algorithmic Space Weather Evaluation
flare = HeliophysicsAlgorithms.classify_goes_solar_flare(5.4e-5)  # M5.4 flare
storm = HeliophysicsAlgorithms.evaluate_geomagnetic_storm(kp_index=7.3, dst_index_nt=-145.0)  # G3 / Intense storm
print(f"Solar Flare Class: {flare['goes_flare_class']} (Severity: {flare['severity_index']})")
print(f"Geomagnetic Storm: {storm['noaa_g_scale']} (Dst: {storm['dst_storm_status']})")

In [ ]:
# 3. Launch Dedicated Worker Service with Public ngrok Tunnel
from scripts.worker_bootstrap import create_worker_app
import threading
import uvicorn

app = create_worker_app("heliophysics", device=device)

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8002, log_level="warning")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("✅ Heliophysics Worker running locally on port 8002")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8002, "http")
    print("\n🌐 HELIOPHYSICS PUBLIC WORKER TUNNEL:", tunnel.public_url)
    print("👉 Register this endpoint with ICARUS_CORE.")
except Exception as e:
    print("Tunnel notice:", e)